# A01 · Pandas — Working with Business Data

**IMB 2026 · MADA · Workshop A01 (Wed 17 Jun 2026, 9:00–13:00)**

Welcome to your first day as a data analyst at **Adriatica**, an e-commerce retailer
selling across the Adriatic region. The CEO wants to know how the business is *really*
doing — but all you have is a messy export from the order system.

By the end of this workshop you will be able to:

- load CSV and Excel files into pandas and inspect them,
- filter, sort, and create calculated columns,
- diagnose and fix dirty data (missing values, duplicates, wrong types, inconsistent labels),
- answer real management questions with `groupby`, pivot tables, and `merge`,
- draw and critically read basic charts,
- read a Python error message without panic.

## 0 · Working in Colab — 10 minutes

1. **File → Save a copy in Drive** — do this *now*, so your work is yours.
2. A notebook is a sequence of **cells**: text cells (like this one) and code cells. Run a code cell with the ▶ button or **Ctrl+Enter**.
3. Exercises are marked **✏️ TODO** — you type the code. Below each exercise there is a collapsed **💡 Show solution**. Etiquette: *try first*, then ask **Gemini Learn Mode** for a hint (spark icon → Learn Mode), and only then open the solution.
4. If a cell errors: read the **last line** of the message first, then find the line it points at. Errors are normal; they are how analysts talk to computers.

In [ ]:
print("Hello, Adriatica!")  # run me: Ctrl+Enter

## 1 · First contact with the data

A pandas **DataFrame** is a spreadsheet with superpowers: rows, named columns, and
every operation repeatable in code. The cell below defines a small loader and reads
`orders.csv` — one row per order.

In [ ]:
import os
import pandas as pd

DATA_BASE_URL = ""  # INSTRUCTOR: paste the published data folder URL here before class (see instructor-notes.md).

def load_file(name, reader=pd.read_csv, **kw):
    """Load a course data file: published URL -> local file -> Colab upload."""
    if DATA_BASE_URL:
        return reader(DATA_BASE_URL.rstrip("/") + "/" + name, **kw)
    if os.path.exists(name):
        return reader(name, **kw)
    try:
        from google.colab import files
        print(f"'{name}' not found - please upload it now (from this workshop's data/ folder).")
        files.upload()
        return reader(name, **kw)
    except ImportError as exc:
        raise FileNotFoundError(name) from exc

orders = load_file("orders.csv")
print("rows, columns:", orders.shape)
orders.head()

Three inspection tools you will use on *every* dataset you ever meet:

- `head()` — what does it look like?
- `info()` — column types and missing values
- `describe()` — quick statistics for the numeric columns

Run the next cell and answer for yourself: *how many rows? which columns are text,
which are numbers? anything suspicious?*

In [ ]:
orders.info()
orders.describe()

**Python aside #1.** `orders` is a *variable* — a name we chose. `orders.head()`
*calls a method* on it. `load_file("orders.csv")` passed an *argument* to a function.
That is 80% of the Python you need today.

In [ ]:
# ✏️ TODO: load the product catalogue from the Excel file "products.xlsx" into a variable called products, and show its first rows
# Hint: load_file can use a different reader: load_file("products.xlsx", reader=pd.read_excel)



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
products = load_file("products.xlsx", reader=pd.read_excel)
products.head()
```

</details>

## 2 · Questions you can already answer

Each order row has `quantity` and `unit_price` — but the CEO thinks in **revenue**.
A calculated column is one line of code.

In [ ]:
# ✏️ TODO: create a new column orders["revenue"] = quantity × unit_price, then look at head() to check it
# Hint: multiply two columns: orders["quantity"] * orders["unit_price"]



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders["revenue"] = orders["quantity"] * orders["unit_price"]
orders.head(3)
```

</details>

**Python aside #2.** `orders["revenue"] > 500` produces a column of `True`/`False`
values — a *boolean mask*. Putting a mask inside `orders[...]` keeps only the `True` rows.
That is filtering, and you will use it daily.

In [ ]:
# ✏️ TODO: filter: (a) orders with revenue above 500 EUR, (b) orders in the "Electronics" category — print how many of each
# Hint: mask = orders["category"] == "Electronics"; then orders[mask]



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
big_orders = orders[orders["revenue"] > 500]
electronics = orders[orders["category"] == "Electronics"]
print(len(big_orders), "orders above 500 EUR")
print(len(electronics), "Electronics orders")
```

</details>

In [ ]:
# ✏️ TODO: show the 10 largest orders by revenue
# Hint: sort_values("revenue", ascending=False) and head(10)



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders.sort_values("revenue", ascending=False).head(10)
```

</details>

**🎯 Stretch (if you are ahead):** merge `products` (it has `unit_cost`) onto `orders`
and compute a `margin` column = revenue − quantity × unit_cost. Which category has the
best total margin?

## 3 · Reality check: dirty data

Now the customer file — and your first taste of real-world data. **Dirty data is the
norm, not the exception.** Every fix below is a *business decision*, not just code.

In [ ]:
# ✏️ TODO: load "customers.csv" into a variable customers and count the missing values in every column
# Hint: customers.isna().sum()



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
customers = load_file("customers.csv")
customers.isna().sum()
```

</details>

**✍️ Your answer** — ~40 customers have no e-mail and ~25 have no country. For each: drop the rows, fill the gaps, or keep as-is? What would each choice do to a mailing campaign vs. a revenue-by-country report?

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ TODO: some customers appear twice — count and remove duplicate rows, printing the row count before and after
# Hint: customers.duplicated().sum(), then drop_duplicates()



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
print("before:", len(customers))
customers = customers.drop_duplicates()
print("after: ", len(customers))
```

</details>

In [ ]:
# ✏️ TODO: check customers.dtypes — signup_date is stored as text; convert it to a real date
# Hint: pd.to_datetime(customers["signup_date"])



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
customers.dtypes
```

</details>

Look at `customers["country"].value_counts()` — Slovenia appears as `Slovenia`,
`SI` **and** `slovenia`; Italy also as `Italia`; Croatia as `HR`. To a computer those are
five different countries. A *mapping dictionary* + `replace` fixes labels; `fillna`
handles the missing ones.

In [ ]:
# ✏️ TODO: normalise the country labels using the mapping {"SI": "Slovenia", "slovenia": "Slovenia", "Italia": "Italy", "HR": "Croatia"}, fill missing countries with "Unknown", and show value_counts()
# Hint: customers["country"].replace(mapping) then .fillna("Unknown")



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
mapping = {"SI": "Slovenia", "slovenia": "Slovenia", "Italia": "Italy", "HR": "Croatia"}
customers["country"] = customers["country"].replace(mapping)
customers["country"] = customers["country"].fillna("Unknown")
customers["country"].value_counts()
```

</details>

One more skeleton in the closet: the **orders** file contains some duplicated
rows too (the export ran twice for a few orders). If we don't remove them, every
revenue number we report will be silently wrong.

In [ ]:
# ✏️ TODO: count duplicate rows in orders, remove them, and print the new row count
# Hint: orders.duplicated().sum() / orders.drop_duplicates()



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
print("duplicate order rows:", orders.duplicated().sum())
orders = orders.drop_duplicates()
print("orders after cleaning:", len(orders))
```

</details>

## 4 · Management questions at scale

The pattern for the rest of your career: **management question → groupby**.

> *"How is revenue developing month by month?"* → group orders by month, sum revenue.

In [ ]:
# ✏️ TODO: compute monthly revenue: convert order_date to datetime, derive a month column with .dt.to_period("M"), then groupby month and sum revenue
# Hint: orders.groupby("month")["revenue"].sum()



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders["order_date"].dt.to_period("M")
monthly_revenue = orders.groupby("month")["revenue"].sum()
monthly_revenue
```

</details>

**📌 Remember this number.** Note your **December 2024** revenue and the **total**
for the year — you will meet them again in workshops A04 and A05 (no spoilers).

**✍️ Your answer** — In one sentence for the CEO: what happened to revenue over the year?

> *(double-click this cell and write your answer here)*

> *"Which market makes us the most money?"* — `orders` has no country column;
`customers` does. Two tables, one question → **merge** (pandas' version of a join).

In [ ]:
# ✏️ TODO: merge orders with customers[["customer_id", "country"]] on customer_id, then compute revenue by country, sorted descending
# Hint: orders.merge(customers[["customer_id", "country"]], on="customer_id")



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders_c = orders.merge(customers[["customer_id", "country"]], on="customer_id")
revenue_by_country = orders_c.groupby("country")["revenue"].sum().sort_values(ascending=False)
revenue_by_country
```

</details>

**✍️ Your answer** — Which two markets would you prioritise next year, and why?

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ TODO: average order value by customer segment (consumer vs business) using a pivot table
# Hint: pivot_table(values="revenue", index="segment", aggfunc="mean")



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders_s = orders.merge(customers[["customer_id", "segment"]], on="customer_id")
orders_s.pivot_table(values="revenue", index="segment", aggfunc="mean").round(2)
```

</details>

## 5 · Show, don't tell

Numbers convince analysts; **charts convince boards**. pandas plots directly from
a DataFrame or Series.

In [ ]:
# ✏️ TODO: draw the monthly revenue as a line chart (add a title and a y-axis label)
# Hint: monthly_revenue.plot(kind="line", marker="o", title=...)



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
import matplotlib.pyplot as plt
monthly_revenue.plot(kind="line", marker="o", title="Adriatica monthly revenue (FY 2024/25)")
plt.ylabel("revenue (EUR)")
plt.show()
```

</details>

In [ ]:
# ✏️ TODO: draw the top-10 products by total revenue as a horizontal bar chart
# Hint: groupby product_name, sum revenue, .nlargest(10), .plot(kind="barh")



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
top10_products = orders.groupby("product_name")["revenue"].sum().nlargest(10)
top10_products.plot(kind="barh", title="Top 10 products by revenue")
plt.xlabel("revenue (EUR)")
plt.show()
```

</details>

In [ ]:
# ✏️ TODO: draw a histogram of order values (try bins=40)
# Hint: orders["revenue"].plot(kind="hist", bins=40)



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders["revenue"].plot(kind="hist", bins=40, title="Order value distribution")
plt.xlabel("order value (EUR)")
plt.show()
```

</details>

**✍️ Your answer** — Pick one of your three charts: what does it NOT tell you? (Think: averages hide distributions; a rising line hides *why*; axes can mislead.)

> *(double-click this cell and write your answer here)*

## 6 · Wrap-up

You walked in with a messy export. You now have: clean tables, a revenue trend,
your top markets and products, and three board-ready charts — all *repeatable*:
rerun the notebook on next month's export and everything updates.

**This afternoon (A02):** you can now *describe what happened*. Next we **predict
what happens next** — which first-time buyers will come back? Same data, new superpower.

**🎯 Self-paced stretch:**

In [ ]:
# ✏️ TODO: write a function country_summary(name) that returns the number of orders, total revenue and average order value for one country
# Hint: def country_summary(name): sub = orders_c[orders_c["country"] == name] ...



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
def country_summary(name):
    sub = orders_c[orders_c["country"] == name]
    return {"orders": len(sub), "revenue": round(sub["revenue"].sum(), 2),
            "avg_order": round(sub["revenue"].mean(), 2)}

country_summary("Slovenia")
```

</details>